# Brazilian Marketplace (Olist) — Business Analysis

Olist is a Brazilian marketplace connecting small businesses to a wide network of online sales channels. This notebook analyzes ~100,000 orders placed on the Olist Store between 2016 and 2018, using the public [Olist Brazilian E-commerce Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle, CC BY-NC-SA 4.0).

**Business question:** How can Olist optimize its operational and marketing strategy to improve profitability, increase customer satisfaction, and reduce delivery times, based on historical sales data, customer reviews, and logistics metrics?

**Method:** SQL on Databricks — data modeling, cleaning, revenue analysis, AOV, CLV, RFM segmentation, churn rate, and correlation studies.

## 1. Setup & Schema Overview

The 9 source CSVs were loaded as tables in `workspace.brazilian_marketplace`. Before analyzing, let's confirm the schema of each table matches the dataset documentation.

In [0]:
SELECT table_name, column_name, data_type, ordinal_position
FROM workspace.information_schema.columns
WHERE table_schema = 'brazilian_marketplace'
ORDER BY table_name, ordinal_position;

The schema matches the dataset documentation for 7 of 9 tables. Two exceptions worth noting:

- `product_category_name_translation` carries an extra `dummy_column` (a constant value on every row) — a leftover from the original BigQuery load, which needed at least one numeric-looking column. Excluded from all queries going forward.
- `reviews` only has `review_id`, `order_id`, `review_score` (not the full 7-column review dataset) — we're using the pre-cleaned version of the reviews file, which already dropped free-text comments and dates. Sufficient here, since this analysis only needs the review score.

Row counts per table, as a sanity check against the documented dataset size:


In [0]:
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM workspace.brazilian_marketplace.customers
UNION ALL SELECT 'geolocation', COUNT(*) FROM workspace.brazilian_marketplace.geolocation
UNION ALL SELECT 'order_items', COUNT(*) FROM workspace.brazilian_marketplace.order_items
UNION ALL SELECT 'orders', COUNT(*) FROM workspace.brazilian_marketplace.orders
UNION ALL SELECT 'payments', COUNT(*) FROM workspace.brazilian_marketplace.payments
UNION ALL SELECT 'reviews', COUNT(*) FROM workspace.brazilian_marketplace.reviews
UNION ALL SELECT 'products', COUNT(*) FROM workspace.brazilian_marketplace.products
UNION ALL SELECT 'sellers', COUNT(*) FROM workspace.brazilian_marketplace.sellers
UNION ALL SELECT 'product_category_name_translation', COUNT(*) FROM workspace.brazilian_marketplace.product_category_name_translation;

All counts match the documented Olist dataset sizes (e.g. 99,441 customers and orders, 112,650 order items, 1,000,163 geolocation records) — the load is clean.

## 2. Primary Key Identification

For each table, a column is a primary key candidate if `COUNT(*) = COUNT(DISTINCT column)` — every value is present and unique.

In [0]:
SELECT 'customers' AS table_name, 'customer_id' AS candidate_column,
       COUNT(*) AS total_rows, COUNT(DISTINCT customer_id) AS distinct_non_null,
       COUNT(*) = COUNT(DISTINCT customer_id) AS is_primary_key
FROM workspace.brazilian_marketplace.customers
UNION ALL
SELECT 'orders', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.orders
UNION ALL
SELECT 'products', 'product_id', COUNT(*), COUNT(DISTINCT product_id), COUNT(*) = COUNT(DISTINCT product_id)
FROM workspace.brazilian_marketplace.products
UNION ALL
SELECT 'sellers', 'seller_id', COUNT(*), COUNT(DISTINCT seller_id), COUNT(*) = COUNT(DISTINCT seller_id)
FROM workspace.brazilian_marketplace.sellers
UNION ALL
SELECT 'reviews', 'review_id', COUNT(*), COUNT(DISTINCT review_id), COUNT(*) = COUNT(DISTINCT review_id)
FROM workspace.brazilian_marketplace.reviews
UNION ALL
SELECT 'product_category_name_translation', 'product_category_name', COUNT(*), COUNT(DISTINCT product_category_name), COUNT(*) = COUNT(DISTINCT product_category_name)
FROM workspace.brazilian_marketplace.product_category_name_translation
UNION ALL
SELECT 'order_items', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.order_items
UNION ALL
SELECT 'payments', 'order_id', COUNT(*), COUNT(DISTINCT order_id), COUNT(*) = COUNT(DISTINCT order_id)
FROM workspace.brazilian_marketplace.payments
UNION ALL
SELECT 'geolocation', 'geolocation_zip_code_prefix', COUNT(*), COUNT(DISTINCT geolocation_zip_code_prefix), COUNT(*) = COUNT(DISTINCT geolocation_zip_code_prefix)
FROM workspace.brazilian_marketplace.geolocation;